In [0]:
# Gold Layer: Customer Lifetime Value (CLV)

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, sum as spark_sum, count, max as spark_max, min as spark_min,
    datediff, current_date, when, lit, current_timestamp,
    round as spark_round, coalesce
)

spark = SparkSession.builder.getOrCreate()
GOLD_TABLE = "ecomstore.ecomlake.gold_customer_lifetime_value"

orders = spark.read.table("ecomstore.ecomlake.silver_orders")
customers = spark.read.table("ecomstore.ecomlake.silver_customers_scd").filter(col("is_current") == True)
returns = spark.read.table("ecomstore.ecomlake.silver_returns")

# 1. PRE-AGGREGATE RETURNS
order_level_returns = (
    returns
    .groupBy("order_id")
    .agg(
        count("return_id").alias("items_returned_count"),
        spark_sum("refund_amount").alias("order_refund_total")
    )
)

# 2. JOIN ORDERS WITH AGGREGATED RETURNS
orders_with_returns = orders.alias("o").join(order_level_returns.alias("r"), on="order_id", how="left")

# 3. CUSTOMER LEVEL AGGREGATION & RFM MATH
customer_metrics = (
    orders_with_returns
    .groupBy("customer_id")
    .agg(
        count("order_id").alias("total_orders"),
        count(when(col("status") == "delivered", True)).alias("successful_orders"),
        spark_sum("final_amount").alias("gross_spent"),
        spark_max("order_date").alias("last_order_date"),
        spark_min("order_date").alias("first_order_date"),
        
        # Safely sum returns
        spark_sum(coalesce(col("items_returned_count"), lit(0))).alias("total_items_returned"),
        spark_sum(coalesce(col("order_refund_total"), lit(0.0))).alias("total_refunded")
    )
    # Financial & Temporal Calculations
    .withColumn("net_revenue", spark_round(col("gross_spent") - col("total_refunded"), 2))
    .withColumn("avg_order_value", 
        when(col("successful_orders") > 0, spark_round(col("net_revenue") / col("successful_orders"), 2))
        .otherwise(0.0)
    )
    .withColumn("days_since_last_order", datediff(current_date(), col("last_order_date")))
    .withColumn("customer_tenure_days", datediff(col("last_order_date"), col("first_order_date")))
    .withColumn("return_rate_pct", 
        when(col("total_orders") > 0, spark_round((col("total_items_returned") / col("total_orders")) * 100, 2))
        .otherwise(0.0)
    )
    # RFM-based segmentation (Optimized for INR thresholds)
    .withColumn("clv_segment",
        when(col("net_revenue") >= 500000, "Platinum")
        .when(col("net_revenue") >= 300000, "Gold")
        .when(col("net_revenue") >= 200000, "Silver")
        .otherwise("Bronze")
    )
)

# 4. ENRICH & WRITE TO GOLD (Optimized)
final_clv = (
    customer_metrics.alias("m")
    .join(
        customers.select("customer_id", "first_name", "last_name", "city", "customer_tier").alias("c"),
        on="customer_id",
        how="left"
    )
    .withColumn("_gold_processed_at", current_timestamp())
)

(
    final_clv
    .coalesce(1)
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .option("delta.autoOptimize.optimizeWrite", "true")
    .option("delta.autoOptimize.autoCompact", "true")
    .saveAsTable(GOLD_TABLE)
)